# B3-swap verified-patch v1 — test-only Devign + REVEAL
Shared evaluator and locked runtime; B3 validation thresholds inherited. Test labels remain local.


In [1]:
from pathlib import Path
import base64, hashlib, json, os, sys, zipfile, shutil, subprocess, time, platform
from datetime import datetime, timezone
WORK=Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SESSION_STARTED=time.time()
for name, encoded in {'grace_runtime_contract.py': 'aW1wb3J0IGNvbnRleHRsaWIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YQppbXBvcnQganNvbgppbXBvcnQgbXVsdGlwcm9jZXNzaW5nCmltcG9ydCBvcwppbXBvcnQgcGxhdGZvcm0KaW1wb3J0IHJlCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCkVYUEVDVEVEX0NPTlRSQUNUX1NIQTI1NiA9ICc0YzljM2JjZmY1ZTlmZDMyNTMyZDdlZGY4NWRmN2E5NjFlNDVlYzljNjRiZWUzZGY4ODMzZDk4NmEyZWIyNzZlJwpDT05UUkFDVF9QQVRIID0gUGF0aCgnL2thZ2dsZS93b3JraW5nL2luZmVyZW5jZV9ydW50aW1lX2xvY2suanNvbicpCgpkZWYgZGlnZXN0KHBhdGgpOgogICAgaCA9IGhhc2hsaWIuc2hhMjU2KCkKICAgIHdpdGggUGF0aChwYXRoKS5vcGVuKCdyYicpIGFzIGY6CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBmLnJlYWQoMTAyNCAqIDEwMjQpLCBiJycpOiBoLnVwZGF0ZShibG9jaykKICAgIHJldHVybiBoLmhleGRpZ2VzdCgpCgpkZWYgbG9hZF9jb250cmFjdChwYXRoPUNPTlRSQUNUX1BBVEgpOgogICAgcmF3ID0gUGF0aChwYXRoKS5yZWFkX2J5dGVzKCkKICAgIGlmIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkgIT0gRVhQRUNURURfQ09OVFJBQ1RfU0hBMjU2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1J1bnRpbWUgY29udHJhY3QgZmlsZSBTSEEtMjU2IG1pc21hdGNoJykKICAgIGMgPSBqc29uLmxvYWRzKHJhdy5kZWNvZGUoJ3V0Zi04JykpCiAgICB1bnNpZ25lZCA9IHtrOiB2IGZvciBrLCB2IGluIGMuaXRlbXMoKSBpZiBrICE9ICdpbnRlZ3JpdHknfQogICAgc2VhbCA9IGhhc2hsaWIuc2hhMjU2KGpzb24uZHVtcHModW5zaWduZWQsIHNvcnRfa2V5cz1UcnVlLCBzZXBhcmF0b3JzPSgnLCcsICc6JyksIGVuc3VyZV9hc2NpaT1GYWxzZSkuZW5jb2RlKCd1dGYtOCcpKS5oZXhkaWdlc3QoKQogICAgaWYgc2VhbCAhPSBjWydpbnRlZ3JpdHknXVsnc2hhMjU2J106CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUnVudGltZSBjb250cmFjdCBjYW5vbmljYWwgaW50ZWdyaXR5IG1pc21hdGNoJykKICAgIHJldHVybiBjCgpkZWYgY29tbWFuZChhcmdzKToKICAgIHAgPSBzdWJwcm9jZXNzLnJ1bihhcmdzLCBjaGVjaz1UcnVlLCB0ZXh0PVRydWUsIGNhcHR1cmVfb3V0cHV0PVRydWUpCiAgICByZXR1cm4gcC5zdGRvdXQgKyBwLnN0ZGVycgoKZGVmIGFzc2VydF9ydW50aW1lKGMpOgogICAgYWN0dWFsID0ge25hbWU6IGltcG9ydGxpYi5tZXRhZGF0YS52ZXJzaW9uKG5hbWUpIGZvciBuYW1lIGluIGNbJ3BhY2thZ2VzJ119CiAgICBpZiBhY3R1YWwgIT0gY1sncGFja2FnZXMnXTogcmFpc2UgUnVudGltZUVycm9yKGYnUGFja2FnZSBtaXNtYXRjaDoge2FjdHVhbH0nKQogICAgaWYgcGxhdGZvcm0uc3lzdGVtKCkgIT0gJ0xpbnV4JyBvciAnLicuam9pbihtYXAoc3RyLCBzeXMudmVyc2lvbl9pbmZvWzoyXSkpICE9IGNbJ2J1aWxkJ11bJ3B5dGhvbl9mYW1pbHknXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0NvbnRyYWN0IHJlcXVpcmVzIEthZ2dsZSBMaW51eC9QeXRob24gMy4xMycpCiAgICBncHUgPSBjb21tYW5kKFsnbnZpZGlhLXNtaScsICctLXF1ZXJ5LWdwdT1pbmRleCx1dWlkLG5hbWUsbWVtb3J5LmZyZWUsbWVtb3J5LnRvdGFsLGRyaXZlcl92ZXJzaW9uJywgJy0tZm9ybWF0PWNzdixub2hlYWRlciddKQogICAgbGluZXMgPSBbcyBmb3IgcyBpbiBncHUuc3BsaXRsaW5lcygpIGlmIHMuc3RyaXAoKV0KICAgIGlmIGxlbihsaW5lcykgIT0gY1snZ3B1J11bJ2RldmljZV9jb3VudCddIG9yIGFueSgnVDQnIG5vdCBpbiBzIGZvciBzIGluIGxpbmVzKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZidDb250cmFjdCByZXF1aXJlcyAyIFQ0IEdQVXM6IHtncHV9JykKICAgIGRpc3QgPSBpbXBvcnRsaWIubWV0YWRhdGEuZGlzdHJpYnV0aW9uKCdsbGFtYS1jcHAtcHl0aG9uJykKICAgIG9yaWdpbiA9IGpzb24ubG9hZHMoZGlzdC5yZWFkX3RleHQoJ2RpcmVjdF91cmwuanNvbicpIG9yICd7fScpCiAgICBpZiBvcmlnaW4uZ2V0KCdhcmNoaXZlX2luZm8nLCB7fSkuZ2V0KCdoYXNoZXMnLCB7fSkuZ2V0KCdzaGEyNTYnKSAhPSBjWydiYWNrZW5kX3NvdXJjZSddWydzZGlzdF9zaGEyNTYnXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0JhY2tlbmQgaW5zdGFsbGF0aW9uIGlzIG5vdCBmcm9tIHRoZSB2ZXJpZmllZCBzb3VyY2UgYXJjaGl2ZScpCiAgICByZXR1cm4gYWN0dWFsLCBncHUKCmRlZiBwcm92ZW5hbmNlKGMsIHJ1bm5lciwgdGFyZ2V0KToKICAgIHBhY2thZ2VzLCBncHUgPSBhc3NlcnRfcnVudGltZShjKQogICAgZGlzdCA9IGltcG9ydGxpYi5tZXRhZGF0YS5kaXN0cmlidXRpb24oJ2xsYW1hLWNwcC1weXRob24nKQogICAgYmluYXJpZXMgPSB7fQogICAgZm9yIGVudHJ5IGluIGRpc3QuZmlsZXMgb3IgW106CiAgICAgICAgaWYgc3RyKGVudHJ5KS5lbmRzd2l0aCgoJy5zbycsICcuZGxsJywgJy5keWxpYicpKToKICAgICAgICAgICAgcCA9IFBhdGgoZGlzdC5sb2NhdGVfZmlsZShlbnRyeSkpOyBiaW5hcmllc1tzdHIoZW50cnkpXSA9IGRpZ2VzdChwKQogICAgaWYgbm90IGJpbmFyaWVzOiByYWlzZSBSdW50aW1lRXJyb3IoJ05vIGNvbXBpbGVkIGJhY2tlbmQgbGlicmFyeSBmb3VuZCBmb3IgcHJvdmVuYW5jZScpCiAgICByZXN1bHQgPSBkaWN0KHJ1bnRpbWVfY29udHJhY3Rfc2hhMjU2PUVYUEVDVEVEX0NPTlRSQUNUX1NIQTI1NiwKICAgICAgICBydW50aW1lX2NvbnRyYWN0X2ludGVncml0eT1jWydpbnRlZ3JpdHknXVsnc2hhMjU2J10sIHBhY2thZ2VzPXBhY2thZ2VzLAogICAgICAgIHB5dGhvbj1zeXMudmVyc2lvbiwgcGxhdGZvcm09cGxhdGZvcm0ucGxhdGZvcm0oKSwgbGliYz1wbGF0Zm9ybS5saWJjX3ZlcigpLAogICAgICAgIGdwdV9iZWZvcmVfbG9hZD1ncHUsIGNwdV9jb3VudD1tdWx0aXByb2Nlc3NpbmcuY3B1X2NvdW50KCksCiAgICAgICAgbl90aHJlYWRzPW1heChtdWx0aXByb2Nlc3NpbmcuY3B1X2NvdW50KCkgLy8gMiwgMSksIG5fdGhyZWFkc19iYXRjaD1tdWx0aXByb2Nlc3NpbmcuY3B1X2NvdW50KCksCiAgICAgICAgY29uc3RydWN0b3I9Y1snY29uc3RydWN0b3InXSwgcnVubmVyX3NoYTI1Nj1kaWdlc3QocnVubmVyKSwKICAgICAgICBjdWRhX2RldmljZV9vcmRlcj1vcy5lbnZpcm9uLmdldCgnQ1VEQV9ERVZJQ0VfT1JERVInKSwKICAgICAgICBjdWRhX3Zpc2libGVfZGV2aWNlcz1vcy5lbnZpcm9uLmdldCgnQ1VEQV9WSVNJQkxFX0RFVklDRVMnKSwKICAgICAgICBiYWNrZW5kX3NkaXN0X3NoYTI1Nj1jWydiYWNrZW5kX3NvdXJjZSddWydzZGlzdF9zaGEyNTYnXSwgY29tcGlsZWRfYmFja2VuZF9zaGEyNTY9YmluYXJpZXMsCiAgICAgICAgYnVpbGRfZW52aXJvbm1lbnQ9e2s6IG9zLmVudmlyb24uZ2V0KGspIGZvciBrIGluICgnQ01BS0VfQVJHUycsICdDTUFLRV9CVUlMRF9QQVJBTExFTF9MRVZFTCcsICdGT1JDRV9DTUFLRScpfSkKICAgIGZvciBuYW1lLCBhcmdzIGluIFsoJ252Y2MnLCBbJ252Y2MnLCAnLS12ZXJzaW9uJ10pLCAoJ2NvbXBpbGVyJywgWydjYycsICctLXZlcnNpb24nXSksCiAgICAgICAgICAgICAgICAgICAgICAoJ2N4eF9jb21waWxlcicsIFsnYysrJywgJy0tdmVyc2lvbiddKSwgKCdjbWFrZScsIFsnY21ha2UnLCAnLS12ZXJzaW9uJ10pLAogICAgICAgICAgICAgICAgICAgICAgKCdwaXBfZnJlZXplJywgW3N5cy5leGVjdXRhYmxlLCAnLW0nLCAncGlwJywgJ2ZyZWV6ZScsICctLWFsbCddKV06CiAgICAgICAgcmVzdWx0W25hbWVdID0gY29tbWFuZChhcmdzKQogICAgdGFyZ2V0ID0gUGF0aCh0YXJnZXQpOyB0YXJnZXQucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRhcmdldC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MiksIGVuY29kaW5nPSd1dGYtOCcpCiAgICByZXR1cm4gcmVzdWx0CgpkZWYgbG9hZF9sb2NrZWRfbW9kZWwoYywgbW9kZWxfcGF0aCwgdG9rZW5pemVyX21hbmlmZXN0LCBncHVfbGF5ZXJzLCBsb2dfcGF0aCwgbW9kZWxfZGlnZXN0KToKICAgIGltcG9ydCBsbGFtYV9jcHAKICAgIGlmIG1vZGVsX2RpZ2VzdCAhPSBjWydtb2RlbCddWydzaGEyNTYnXTogcmFpc2UgVmFsdWVFcnJvcignRnJvemVuIG1vZGVsIFNIQS0yNTYgbWlzbWF0Y2gnKQogICAgaWYgZ3B1X2xheWVycyAhPSBjWydjb25zdHJ1Y3RvciddWyduX2dwdV9sYXllcnMnXTogcmFpc2UgVmFsdWVFcnJvcignR1BVIGxheWVycyBkaWZmZXIgZnJvbSBsb2NrJykKICAgIGlmICh0b2tlbml6ZXJfbWFuaWZlc3RbJ3Rva2VuaXplciddLCB0b2tlbml6ZXJfbWFuaWZlc3RbJ3Rva2VuaXplcl9yZXZpc2lvbiddLCB0b2tlbml6ZXJfbWFuaWZlc3RbJ3Byb21wdF9idWRnZXRfdG9rZW5zJ10pICE9ICgKICAgICAgICBjWyd0b2tlbml6ZXInXVsncmVwbyddLCBjWyd0b2tlbml6ZXInXVsncmV2aXNpb24nXSwgY1snY29uc3RydWN0b3InXVsnbl9jdHgnXSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVG9rZW5pemVyL2NvbnRleHQgZGlmZmVyIGZyb20gcnVudGltZSBsb2NrJykKICAgIGlmIG5vdCBsbGFtYV9jcHAubGxhbWFfc3VwcG9ydHNfZ3B1X29mZmxvYWQoKTogcmFpc2UgUnVudGltZUVycm9yKCdDVURBIGJhY2tlbmQgdW5hdmFpbGFibGUnKQogICAga3dhcmdzID0gZGljdChjWydjb25zdHJ1Y3RvciddKQogICAga3dhcmdzWydzcGxpdF9tb2RlJ10gPSBnZXRhdHRyKGxsYW1hX2NwcCwga3dhcmdzWydzcGxpdF9tb2RlJ10pCiAgICBrd2FyZ3MudXBkYXRlKG5fdGhyZWFkcz1tYXgobXVsdGlwcm9jZXNzaW5nLmNwdV9jb3VudCgpIC8vIDIsIDEpLCBuX3RocmVhZHNfYmF0Y2g9bXVsdGlwcm9jZXNzaW5nLmNwdV9jb3VudCgpKQogICAgcGF0aCA9IFBhdGgobG9nX3BhdGgpCiAgICB3aXRoIHBhdGgub3BlbigndycsIGVuY29kaW5nPSd1dGYtOCcpIGFzIGxvZywgY29udGV4dGxpYi5yZWRpcmVjdF9zdGRlcnIobG9nKToKICAgICAgICBtb2RlbCA9IGxsYW1hX2NwcC5MbGFtYShtb2RlbF9wYXRoPXN0cihtb2RlbF9wYXRoKSwgdmVyYm9zZT1UcnVlLCAqKmt3YXJncykKICAgIHRleHQgPSBwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0ndXRmLTgnLCBlcnJvcnM9J3JlcGxhY2UnKQogICAgcHJpbnQodGV4dCwgZmx1c2g9VHJ1ZSkKICAgIGZyb20gYjNfbmF0aXZlX3J1bnRpbWUgaW1wb3J0IG5hdGl2ZV9sb2dfY2hlY2sKICAgIHRyeToKICAgICAgICBuYXRpdmVfbG9nX2NoZWNrKHRleHQsIGMpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIG1vZGVsLmNsb3NlKCkKICAgICAgICByYWlzZQogICAgcmV0dXJuIG1vZGVsCg==', 'b3_native_runtime.py': 'aW1wb3J0IHJlCgpkZWYgbmF0aXZlX2xvZ19jaGVjayh0ZXh0LCBjb250cmFjdCk6CiAgICByZXF1aXJlZCA9IGNvbnRyYWN0WydncHUnXVsncmVxdWlyZWRfb2ZmbG9hZGVkX2xheWVycyddCiAgICBwYWlycyA9IHJlLmZpbmRhbGwocidvZmZsb2FkZWRccysoXGQrKVxzKi9ccyooXGQrKVxzK2xheWVycz9ccyt0b1xzK0dQVScsIHRleHQsIHJlLkkpCiAgICBpZiBub3QgYW55KGludChhKSA9PSBpbnQoYikgPT0gcmVxdWlyZWQgZm9yIGEsYiBpbiBwYWlycyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRnVsbCBHUFUgb2ZmbG9hZCBub3QgdmVyaWZpZWQnKQogICAgZm9yIGZpZWxkIGluICgnbl9jdHgnLCduX2JhdGNoJywnbl91YmF0Y2gnKToKICAgICAgICB2YWx1ZXMgPSByZS5maW5kYWxsKHInXGInK2ZpZWxkK3InXHMqPVxzKihcZCspXGInLCB0ZXh0KQogICAgICAgIGlmIG5vdCB2YWx1ZXMgb3IgaW50KHZhbHVlc1stMV0pICE9IGNvbnRyYWN0Wydjb25zdHJ1Y3RvciddW2ZpZWxkXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignRWZmZWN0aXZlICcrZmllbGQrJyBkaWZmZXJzIGZyb20gY29udHJhY3QnKQogICAgZGV2aWNlcyA9IHNldChyZS5maW5kYWxsKHIndXNpbmcgZGV2aWNlIENVREEoXGQrKVxzKlwoW15cbildKlxiVDRcYlteXG4pXSpcKScsIHRleHQsIHJlLkkpKQogICAgZGV2aWNlcy51cGRhdGUocmUuZmluZGFsbChyJ0RldmljZVxzKyhcZCspOlxzK1teXG5dKlxiVDRcYicsIHRleHQpKQogICAgaWYgZGV2aWNlcyAhPSBzZXQobWFwKHN0ciwgcmFuZ2UoY29udHJhY3RbJ2dwdSddWydkZXZpY2VfY291bnQnXSkpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdUd28gZGlzdGluY3QgVDQgZGV2aWNlcyBub3QgdmVyaWZpZWQnKQo=', 'kaggle_inference_test.py': '...', 'b3_test_runtime.py': 'IiIiRmFpbC1jbG9zZWQgQjMgdGVzdCBwYWNrYWdlIGxvYWRlci4gVGFyZ2V0IGxhYmVscyBuZXZlciBlbnRlciB0aGlzIG1vZHVsZS4iIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHJlCmltcG9ydCB6aXBmaWxlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aCwgUHVyZVBvc2l4UGF0aAoKCmRlZiBzaGEocmF3KToKICAgIGlmIGlzaW5zdGFuY2UocmF3LCAoc3RyLCBQYXRoKSk6CiAgICAgICAgcmF3ID0gUGF0aChyYXcpLnJlYWRfYnl0ZXMoKQogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkKCgpkZWYgc2FmZV9uYW1lKG5hbWUpOgogICAgaWYgbm90IG5hbWUgb3IgIlxcIiBpbiBuYW1lIG9yICI6IiBpbiBuYW1lIG9yICJceDAwIiBpbiBuYW1lOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuc2FmZSB0ZXN0IHBhY2thZ2UgcGF0aCIpCiAgICBwID0gUHVyZVBvc2l4UGF0aChuYW1lKQogICAgaWYgcC5pc19hYnNvbHV0ZSgpIG9yIGFueSh4IGluICgiIiwgIi4iLCAiLi4iKSBmb3IgeCBpbiBuYW1lLnNwbGl0KCIvIikpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVuc2FmZSB0ZXN0IHBhY2thZ2UgcGF0aCIpCiAgICByZXR1cm4gbmFtZQoKCmRlZiByZWFkX3ppcChwYXRoKToKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKHBhdGgpIGFzIGFyY2hpdmU6CiAgICAgICAgaWYgYXJjaGl2ZS50ZXN0emlwKCkgaXMgbm90IE5vbmU6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcGFja2FnZSBDUkMgZmFpbHVyZSIpCiAgICAgICAgaW5mb3MgPSBhcmNoaXZlLmluZm9saXN0KCkKICAgICAgICBuYW1lcyA9IFtzYWZlX25hbWUoaS5maWxlbmFtZSkgZm9yIGkgaW4gaW5mb3MgaWYgbm90IGkuaXNfZGlyKCldCiAgICAgICAgaWYgbGVuKG5hbWVzKSAhPSBsZW4oc2V0KG5hbWVzKSk6IHJhaXNlIFZhbHVlRXJyb3IoIkR1cGxpY2F0ZSB0ZXN0IHBhY2thZ2UgbWVtYmVyIikKICAgICAgICBpZiBhbnkoKGkuZXh0ZXJuYWxfYXR0ciA+PiAxNikgJiAwbzE3MDAwMCA9PSAwbzEyMDAwMCBmb3IgaSBpbiBpbmZvcyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlN5bWxpbmsgdGVzdCBwYWNrYWdlIG1lbWJlciIpCiAgICAgICAgcmV0dXJuIHtuYW1lOiBhcmNoaXZlLnJlYWQobmFtZSkgZm9yIG5hbWUgaW4gbmFtZXN9CgoKZGVmIGxvYWRfbWFuaWZlc3QocGF0aCwgZXhwZWN0ZWRfc2hhMjU2KToKICAgIHJhdyA9IFBhdGgocGF0aCkucmVhZF9ieXRlcygpCiAgICBpZiBzaGEocmF3KSAhPSBleHBlY3RlZF9zaGEyNTY6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcHJvbXB0IG1hbmlmZXN0IGhhc2ggbWlzbWF0Y2giKQogICAgcm9vdCA9IGpzb24ubG9hZHMocmF3KQogICAgaWYgcm9vdC5nZXQoInN0YXR1cyIpICE9ICJTRUFMRURfVEVTVF9ERUxJVkVSWSI6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgcGFja2FnZSBub3Qgc2VhbGVkIikKICAgIGlmIHJvb3QuZ2V0KCJhcm0iKSBub3QgaW4gKCJCMyIsICJCMy1zd2FwIiwgIkIzLWZsaXAiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJVbnN1cHBvcnRlZCB0ZXN0IGFybSIpCiAgICBpZiAocm9vdC5nZXQoIm1ldGhvZF9pZCIpLCByb290LmdldCgic2VlZCIpLCByb290LmdldCgiaW5mZXJlbmNlX3NwbGl0IikpICE9ICgKICAgICAgICAgICAgImIzLXZlcmlmaWVkLXBhdGNoLXYxIiwgNDIsICJ0ZXN0Iik6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBwYWNrYWdlIHNjb3BlIG1pc21hdGNoIikKICAgIGlmIHJvb3QuZ2V0KCJsYWJlbHNfaW5fa2FnZ2xlX2lucHV0IikgaXMgbm90IEZhbHNlOiByYWlzZSBWYWx1ZUVycm9yKCJUYXJnZXQgbGFiZWxzIG11c3QgYmUgaGVsZCBvdXQgb2YgS2FnZ2xlIGlucHV0IikKICAgIHJldHVybiByb290CgoKZGVmIHZhbGlkYXRlX2FyY2hpdmUocGF0aCwgcm9vdCk6CiAgICByYXcgPSBQYXRoKHBhdGgpLnJlYWRfYnl0ZXMoKQogICAgaWYgc2hhKHJhdykgIT0gcm9vdFsia2FnZ2xlX2lucHV0X2FyY2hpdmVfc2hhMjU2Il06IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3QgaW5wdXQgYXJjaGl2ZSBjaGVja3N1bSBtaXNtYXRjaCIpCiAgICBtZW1iZXJzID0gcmVhZF96aXAocGF0aCkKICAgIGlmIHNldChtZW1iZXJzKSAhPSBzZXQocm9vdFsia2FnZ2xlX2lucHV0X21lbWJlcnNfc2hhMjU2Il0pOiByYWlzZSBWYWx1ZUVycm9yKCJUZXN0IGlucHV0IHBhY2thZ2UgaW52ZW50b3J5IG1pc21hdGNoIikKICAgIGZvciBuYW1lLCBleHBlY3RlZCBpbiByb290WyJrYWdnbGVfaW5wdXRfbWVtYmVyc19zaGEyNTYiXS5pdGVtcygpOgogICAgICAgIGlmIHNoYShtZW1iZXJzW25hbWVdKSAhPSBleHBlY3RlZDogcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBpbnB1dCBtZW1iZXIgaGFzaCBtaXNtYXRjaDogIiArIG5hbWUpCiAgICBpZiBhbnkocmUuc2VhcmNoKHIiKF58LykoPzpsYWJlbHMoPzpcLmpzb258XC5qc29ubCk/fHRlc3RfbGFiZWxzXC5qc29uKSQiLCBuLCByZS5JKSBmb3IgbiBpbiBtZW1iZXJzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJMYWJlbCBhcnRpZmFjdCBmb3VuZCBpbiBLYWdnbGUgaW5mZXJlbmNlIHBhY2thZ2UiKQogICAgcmV0dXJuIG1lbWJlcnMKCgpkZWYgcmVhZF9pbnB1dF9kaXJlY3RvcnkocGF0aCk6CiAgICBiYXNlID0gUGF0aChwYXRoKQogICAgaWYgbm90IGJhc2UuaXNfZGlyKCk6IHJhaXNlIFZhbHVlRXJyb3IoIkV4cGFuZGVkIHRlc3QgaW5wdXQgZGlyZWN0b3J5IGlzIG1pc3NpbmciKQogICAgbWVtYmVycyA9IHt9CiAgICBmb3IgaXRlbSBpbiBzb3J0ZWQoYmFzZS5yZ2xvYigiKiIpKToKICAgICAgICBpZiBpdGVtLmlzX3N5bWxpbmsoKTogcmFpc2UgVmFsdWVFcnJvcigiU3ltbGluayBpbiBleHBhbmRlZCB0ZXN0IGlucHV0IikKICAgICAgICBpZiBpdGVtLmlzX2ZpbGUoKToKICAgICAgICAgICAgbmFtZSA9IGl0ZW0ucmVsYXRpdmVfdG8oYmFzZSkuYXNfcG9zaXgoKQogICAgICAgICAgICBzYWZlX25hbWUobmFtZSkKICAgICAgICAgICAgaWYgbmFtZSBpbiBtZW1iZXJzOiByYWlzZSBWYWx1ZUVycm9yKCJEdXBsaWNhdGUgZXhwYW5kZWQgdGVzdCBpbnB1dCBtZW1iZXIiKQogICAgICAgICAgICBtZW1iZXJzW25hbWVdID0gaXRlbS5yZWFkX2J5dGVzKCkKICAgIHJldHVybiBtZW1iZXJzCgoKZGVmIHZhbGlkYXRlX21lbWJlcnMobWVtYmVycywgcm9vdCk6CiAgICBpZiBzZXQobWVtYmVycykgIT0gc2V0KHJvb3RbImthZ2dsZV9pbnB1dF9tZW1iZXJzX3NoYTI1NiJdKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUZXN0IGlucHV0IG1lbWJlciBpbnZlbnRvcnkgbWlzbWF0Y2giKQogICAgZm9yIG5hbWUsIGV4cGVjdGVkIGluIHJvb3RbImthZ2dsZV9pbnB1dF9tZW1iZXJzX3NoYTI1NiJdLml0ZW1zKCk6CiAgICAgICAgaWYgc2hhKG1lbWJlcnNbbmFtZV0pICE9IGV4cGVjdGVkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUZXN0IGlucHV0IG1lbWJlciBjaGVja3N1bSBtaXNtYXRjaDogIiArIG5hbWUpCiAgICBpZiBhbnkocmUuc2VhcmNoKHIiKF58LykoPzpsYWJlbHMoPzpcLmpzb258XC5qc29ubCk/fHRlc3RfbGFiZWxzXC5qc29uKSQiLCBuLCByZS5JKSBmb3IgbiBpbiBtZW1iZXJzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJMYWJlbCBhcnRpZmFjdCBmb3VuZCBpbiBLYWdnbGUgaW5mZXJlbmNlIHBhY2thZ2UiKQogICAgcmV0dXJuIG1lbWJlcnMKCgpkZWYgcmVhZF9kYXRhc2V0X3Jvd3MobWVtYmVycywgcm9vdCwgZGF0YXNldCk6CiAgICBpbmZvID0gcm9vdFsiZGF0YXNldHMiXVtkYXRhc2V0XQogICAgbmFtZSA9IGYicHJvbXB0cy97ZGF0YXNldH0uanNvbmwiCiAgICByYXcgPSBtZW1iZXJzW25hbWVdCiAgICBpZiBzaGEocmF3KSAhPSBpbmZvWyJwcm9tcHRzX3NoYTI1NiJdOiByYWlzZSBWYWx1ZUVycm9yKCJQcm9tcHQgc2V0IGhhc2ggbWlzbWF0Y2giKQogICAgcm93cyA9IFtqc29uLmxvYWRzKGxpbmUpIGZvciBsaW5lIGluIHJhdy5kZWNvZGUoInV0Zi04Iikuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KICAgIGlmIGxlbihyb3dzKSAhPSBpbmZvWyJleHBlY3RlZCJdIG9yIGxlbih7ci5nZXQoImlkIikgZm9yIHIgaW4gcm93c30pICE9IGxlbihyb3dzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUZXN0IGNlbnN1cyBoYXMgbWlzc2luZy9kdXBsaWNhdGUgSURzIikKICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICBpZiByb290Wydhcm0nXSAhPSAnQjMnIGFuZCByb3cuZ2V0KCdhcm0nKSAhPSByb290Wydhcm0nXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVmFyaWFudCBwcm9tcHQgYXJtIG1pc21hdGNoJykKICAgICAgICBpZiBhbnkoayBpbiByb3cgZm9yIGsgaW4gKCJsYWJlbCIsICJ0YXJnZXQiLCAidGFyZ2V0X2xhYmVsIikpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUYXJnZXQgbGFiZWwgbGVha2VkIGludG8gdGVzdCBwcm9tcHQgcm93IikKICAgICAgICBpZiAocm93LmdldCgiZGF0YXNldCIpLCByb3cuZ2V0KCJzcGxpdCIpLCByb3cuZ2V0KCJtZXRob2RfaWQiKSkgIT0gKGRhdGFzZXQsICJ0ZXN0Iiwgcm9vdFsibWV0aG9kX2lkIl0pOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUZXN0IHByb21wdCBzY29wZSBtaXNtYXRjaCIpCiAgICAgICAgaWYgbm90IHJvd1siaWQiXS5zdGFydHN3aXRoKGRhdGFzZXQgKyAiOnRlc3Q6Iikgb3Igc2hhKHJvd1sicHJvbXB0Il0uZW5jb2RlKCJ1dGYtOCIpKSAhPSByb3dbInByb21wdF9zaGEyNTYiXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBwcm9tcHQgSUQvaGFzaCBtaXNtYXRjaCIpCiAgICByZXR1cm4gcm93cwoKCmRlZiBsb2FkX3RocmVzaG9sZChyb290LCBtZW1iZXJzLCBkYXRhc2V0KToKICAgICIiIlZlcmlmeSB2YWxpZGF0aW9uLWZpdHRlZCB0aHJlc2hvbGQgYnl0ZXMgYW5kIHRoZWlyIGZyb3plbiBtZXRob2QgaWRlbnRpdHkuIiIiCiAgICBuYW1lID0gZiJ0aHJlc2hvbGRfbG9ja3Mve2RhdGFzZXR9X3RocmVzaG9sZF9sb2NrLmpzb24iCiAgICByYXcgPSBtZW1iZXJzW25hbWVdCiAgICBpZiBzaGEocmF3KSAhPSByb290WyJ0aHJlc2hvbGRfbG9ja19zaGEyNTYiXVtkYXRhc2V0XToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaHJlc2hvbGQgbG9jayBmaWxlIGNoZWNrc3VtIG1pc21hdGNoIikKICAgIGxvY2sgPSBqc29uLmxvYWRzKHJhdy5kZWNvZGUoInV0Zi04IikpCiAgICB1bnNpZ25lZCA9IHtrOiB2IGZvciBrLCB2IGluIGxvY2suaXRlbXMoKSBpZiBrICE9ICJzZWFsX3NoYTI1NiJ9CiAgICBjYW5vbmljYWwgPSBqc29uLmR1bXBzKHVuc2lnbmVkLCBzb3J0X2tleXM9VHJ1ZSwgc2VwYXJhdG9ycz0oIiwiLCAiOiIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICBlbnN1cmVfYXNjaWk9RmFsc2UsIGFsbG93X25hbj1GYWxzZSkuZW5jb2RlKCJ1dGYtOCIpCiAgICBpZiBzaGEoY2Fub25pY2FsKSAhPSBsb2NrLmdldCgic2VhbF9zaGEyNTYiKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaHJlc2hvbGQgbG9jayBjYW5vbmljYWwgc2VhbCBtaXNtYXRjaCIpCiAgICBpZiAobG9jay5nZXQoInN0YXR1cyIpLCBsb2NrLmdldCgiYXJtIiksIGxvY2suZ2V0KCJtZXRob2RfaWQiKSwKICAgICAgICBsb2NrLmdldCgic2VlZCIpLCBsb2NrLmdldCgiZGF0YXNldCIpLCBsb2NrLmdldCgiZml0X3NwbGl0IikpICE9ICgKICAgICAgICAiTE9DS0VEIiwgIkIzIiwgcm9vdFsibWV0aG9kX2lkIl0sIDQyLCBkYXRhc2V0LCAidmFsaWRhdGlvbiIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRocmVzaG9sZCBzY29wZSBtaXNtYXRjaCIpCiAgICBleHBlY3RlZCA9IHsibWV0aG9kX2lkIjogcm9vdFsibWV0aG9kX2lkIl0sCiAgICAgICAgInBvb2xfc2hhMjU2Ijogcm9vdFsiZGF0YXNldHMiXVtkYXRhc2V0XVsicG9vbF9zaGEyNTYiXSwKICAgICAgICAibW9kZWxfc2hhMjU2Ijogcm9vdFsibW9kZWxfc2hhMjU2Il0sCiAgICAgICAgInRva2VuaXplcl9yZXZpc2lvbiI6IHJvb3RbInRva2VuaXplcl9yZXZpc2lvbiJdLAogICAgICAgICJydW50aW1lX2NvbnRyYWN0X3NoYTI1NiI6IHJvb3RbInJ1bnRpbWVfY29udHJhY3Rfc2hhMjU2Il0sCiAgICAgICAgInNjb3JpbmciOiAidW5jb25zdHJhaW5lZF9maXJzdF90b2tlbl9sb2dpdHNfdjEiLAogICAgICAgICJ3b3JrZXJfc2hhMjU2Ijogcm9vdFsidGhyZXNob2xkX2FwcGxpY2F0aW9uX3dvcmtlcl9zaGEyNTYiXX0KICAgIGlmIGFueShsb2NrWyJhcHBsaWNhdGlvbl9pZGVudGl0eSJdLmdldChrKSAhPSB2IGZvciBrLCB2IGluIGV4cGVjdGVkLml0ZW1zKCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRocmVzaG9sZCBhcHBsaWNhdGlvbiBpZGVudGl0eSBtaXNtYXRjaCIpCiAgICB0YXUgPSBsb2NrLmdldCgidGF1X3N0YXIiKQogICAgaWYgdHlwZSh0YXUpIG5vdCBpbiAoZmxvYXQsIGludCkgb3Igbm90IDAuMDEgPD0gdGF1IDw9IDAuOTk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW52YWxpZCBsb2NrZWQgdGhyZXNob2xkIikKICAgIGlmIHRhdSAhPSByb290WyJkYXRhc2V0cyJdW2RhdGFzZXRdWyJ0YXVfc3RhciJdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRocmVzaG9sZCB2YWx1ZSBkaWZmZXJzIGZyb20gc2VhbGVkIHRlc3QgZGVsaXZlcnkiKQogICAgcmV0dXJuIGZsb2F0KHRhdSkKCgpkZWYgcnVuX2lkZW50aXR5KHJvb3QsIGRhdGFzZXQpOgogICAgeCA9IHJvb3RbImRhdGFzZXRzIl1bZGF0YXNldF0KICAgIHJldHVybiB7InJlbGVhc2VfaWQiOiByb290WyJyZWxlYXNlX2lkIl0sICJtZXRob2RfaWQiOiByb290WyJtZXRob2RfaWQiXSwgImFybSI6IHJvb3RbJ2FybSddLAogICAgICAgICAgICAiZGF0YXNldCI6IGRhdGFzZXQsICJzcGxpdCI6ICJ0ZXN0IiwgInNlZWQiOiA0MiwgImNvdW50IjogeFsiZXhwZWN0ZWQiXSwKICAgICAgICAgICAgInByb21wdHNfc2hhMjU2IjogeFsicHJvbXB0c19zaGEyNTYiXSwgInBvb2xfc2hhMjU2IjogeFsicG9vbF9zaGEyNTYiXSwKICAgICAgICAgICAgIm1vZGVsX3NoYTI1NiI6IHJvb3RbIm1vZGVsX3NoYTI1NiJdLCAidG9rZW5pemVyX3JldmlzaW9uIjogcm9vdFsidG9rZW5pemVyX3JldmlzaW9uIl0sCiAgICAgICAgICAgICJydW50aW1lX2NvbnRyYWN0X3NoYTI1NiI6IHJvb3RbInJ1bnRpbWVfY29udHJhY3Rfc2hhMjU2Il0sCiAgICAgICAgICAgICJ0aHJlc2hvbGRfbG9ja19zaGEyNTYiOiByb290WyJ0aHJlc2hvbGRfbG9ja19zaGEyNTYiXVtkYXRhc2V0XSwKICAgICAgICAgICAgIndvcmtlcl9zaGEyNTYiOiByb290WyJ0ZXN0X3dvcmtlcl9zaGEyNTYiXX0KCgpkZWYgY2hlY2tlZF9yZWNvcmRzKG91dHB1dF9yb290LCByb290LCByb3dzLCBkYXRhc2V0LCBzcGxpdD0idGVzdCIsIHNlZWQ9NDIsIGFybT1Ob25lKToKICAgIGFybSA9IHJvb3RbJ2FybSddIGlmIGFybSBpcyBOb25lIGVsc2UgYXJtCiAgICBpZiBhcm0gIT0gcm9vdFsnYXJtJ10gb3Igc3BsaXQgIT0gJ3Rlc3QnIG9yIHNlZWQgIT0gNDI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignUmVzdW1lIGFybS9zcGxpdC9zZWVkIGRpZmZlcnMgZnJvbSBzZWFsZWQgZGVsaXZlcnknKQogICAgYmFzZSA9IFBhdGgob3V0cHV0X3Jvb3QpIC8gZiJzZWVkX3tzZWVkfSIgLyBkYXRhc2V0IC8gc3BsaXQKICAgIHBhdGggPSBiYXNlIC8gInJlY29yZHMuanNvbmwiCiAgICBpZiBub3QgcGF0aC5pc19maWxlKCk6IHJldHVybiAwCiAgICBtYW5pZmVzdF9wYXRoID0gYmFzZSAvICJydW5fbWFuaWZlc3QuanNvbiIKICAgIGlmIG5vdCBtYW5pZmVzdF9wYXRoLmlzX2ZpbGUoKTogcmFpc2UgVmFsdWVFcnJvcigiVGVzdCByZWNvcmRzIGxhY2sgcnVuIGlkZW50aXR5IG1hbmlmZXN0IikKICAgIG1hbmlmZXN0ID0ganNvbi5sb2FkcyhtYW5pZmVzdF9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgIGV4cGVjdGVkX2lkZW50aXR5ID0gcnVuX2lkZW50aXR5KHJvb3QsIGRhdGFzZXQpCiAgICBpZiBtYW5pZmVzdC5nZXQoImIzX3Rlc3RfaWRlbnRpdHkiKSAhPSBleHBlY3RlZF9pZGVudGl0eTogcmFpc2UgVmFsdWVFcnJvcigiVGVzdCByZXN1bWUgaWRlbnRpdHkgbWlzbWF0Y2giKQogICAgbG9va3VwID0ge3JbImlkIl06IHIgZm9yIHIgaW4gcm93c30KICAgIHNlZW4gPSBzZXQoKQogICAgZm9yIGxpbmUgaW4gcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04Iikuc3BsaXRsaW5lcygpOgogICAgICAgIGlmIG5vdCBsaW5lLnN0cmlwKCk6IGNvbnRpbnVlCiAgICAgICAgcmVjb3JkID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgIHVpZCA9IHJlY29yZFsiaWQiXQogICAgICAgIGlmIHVpZCBpbiBzZWVuIG9yIHVpZCBub3QgaW4gbG9va3VwOiByYWlzZSBWYWx1ZUVycm9yKCJEdXBsaWNhdGUvdW5leHBlY3RlZCB0ZXN0IElEIikKICAgICAgICBpZiBhbnkoayBpbiByZWNvcmQgZm9yIGsgaW4gKCJsYWJlbCIsICJ0YXJnZXQiLCAidGFyZ2V0X2xhYmVsIikpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUYXJnZXQgbGFiZWwgbGVha2VkIGludG8gdGVzdCBwcmVkaWN0aW9uIHJlY29yZCIpCiAgICAgICAgcHJvbXB0ID0gbG9va3VwW3VpZF0KICAgICAgICBpZiAocmVjb3JkLmdldCgicHJvbXB0X3NoYTI1NiIpLCByZWNvcmQuZ2V0KCJkYXRhc2V0IiksIHJlY29yZC5nZXQoInNwbGl0IiksCiAgICAgICAgICAgICAgICByZWNvcmQuZ2V0KCJhcm0iKSwgcmVjb3JkLmdldCgic2VlZCIpLCByZWNvcmQuZ2V0KCJ2YWxpZCIpLAogICAgICAgICAgICAgICAgcmVjb3JkLmdldCgibW9kZWxfc2hhMjU2IiksIHJlY29yZC5nZXQoInJ1bnRpbWVfY29udHJhY3Rfc2hhMjU2IikpICE9ICgKICAgICAgICAgICAgICAgIHByb21wdFsicHJvbXB0X3NoYTI1NiJdLCBkYXRhc2V0LCAidGVzdCIsIGFybSwgc2VlZCwgVHJ1ZSwKICAgICAgICAgICAgICAgIHJvb3RbIm1vZGVsX3NoYTI1NiJdLCByb290WyJydW50aW1lX2NvbnRyYWN0X3NoYTI1NiJdKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVGVzdCBwcmVkaWN0aW9uIGlkZW50aXR5IG1pc21hdGNoIikKICAgICAgICBzLCBscDAsIGxwMSA9IHJlY29yZFsicyJdLCByZWNvcmRbImxvZ3Byb2JfMCJdLCByZWNvcmRbImxvZ3Byb2JfMSJdCiAgICAgICAgaWYgYW55KHR5cGUoeCkgbm90IGluIChpbnQsIGZsb2F0KSBvciBub3QgbWF0aC5pc2Zpbml0ZSh4KSBmb3IgeCBpbiAocywgbHAwLCBscDEpKSBvciBub3QgMCA8PSBzIDw9IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkludmFsaWQgdGVzdCBzY29yZSIpCiAgICAgICAgIyBVc2UgYSBzdGFibGUgYnJhbmNoLWZvcm0gc2lnbW9pZCB3aGVuIHRoZSBob3N0IG1hdGggbW9kdWxlIGxhY2tzIGxvZ2FkZGV4cC4KICAgICAgICBkZWx0YSA9IGxwMSAtIGxwMAogICAgICAgIGV4cGVjdGVkID0gMSAvICgxICsgbWF0aC5leHAoLWRlbHRhKSkgaWYgZGVsdGEgPj0gMCBlbHNlIG1hdGguZXhwKGRlbHRhKSAvICgxICsgbWF0aC5leHAoZGVsdGEpKQogICAgICAgIGlmIGFicyhzIC0gZXhwZWN0ZWQpID4gMWUtMTI6IHJhaXNlIFZhbHVlRXJyb3IoIlRlc3Qgc2NvcmUvbG9ncHJvYiBtaXNtYXRjaCIpCiAgICAgICAgdGF1ID0gcm9vdFsiZGF0YXNldHMiXVtkYXRhc2V0XVsidGF1X3N0YXIiXQogICAgICAgIGlmIHJlY29yZC5nZXQoInRocmVzaG9sZF90YXUiKSAhPSB0YXUgb3IgcmVjb3JkLmdldCgidGhyZXNob2xkX3ByZWRpY3Rpb24iKSAhPSBpbnQocyA+PSB0YXUpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUZXN0IGxvY2tlZC10aHJlc2hvbGQgcHJlZGljdGlvbiBtaXNtYXRjaCIpCiAgICAgICAgc2Vlbi5hZGQodWlkKQogICAgcmV0dXJuIGxlbihzZWVuKQo=', 'inference_runtime_lock.json': '...'}.items(): (WORK/name).write_bytes(base64.b64decode(encoded))
sys.path.insert(0,str(WORK))


In [2]:
import glob, os, json, hashlib
from pathlib import Path
from grace_runtime_contract import load_contract, assert_runtime, digest, command
RUNTIME=load_contract()
EXPECTED_MANIFEST_SHA256='3598bf9d005e552690c11cb1e2ae5dd1f4af2a9ac2459e061f5110f04c53952f'
EMBEDDED_MANIFEST_B64='ew0KICAic2NoZW1hIjogImdyYWNlX2IzX3Rlc3RfcmVsZWFzZV92MSIsDQogICJyZWxlYXNlX2lkIjogImIzX3N3YXBfdGVzdF9yZWxlYXNlXzIwMjYxMTEwX3YxIiwNCiAgInN0YXR1cyI6ICJTRUFMRURfVEVTVF9ERUxJVkVSWSIsDQogICJtZXRob2RfaWQiOiAiYjMtdmVyaWZpZWQtcGF0Y2gtdjEiLA0KICAiYXJtIjogIkIzLXN3YXAiLA0KICAic2VlZCI6IDQyLA0KICAiZml0X3NwbGl0IjogInZhbGlkYXRpb24iLA0KICAiaW5mZXJlbmNlX3NwbGl0IjogInRlc3QiLA0KICAibW9kZWxfc2hhMjU2IjogImY2MDUwYjRkYTIyODY5YWEzZmI5ZDk0YmIzMzE0ZTlmYzVhMDI3ZDQ3NDFjMWY5Nzc2MWMzODdkNmIyYjIzMzUiLA0KICAidG9rZW5pemVyX3JldmlzaW9uIjogImIyY2ZmNjQ2ZWI0YmIxZDY4MzU1YzAxYjE4YWUwMmU3Y2Y0MmQxMjAiLA0KICAicnVudGltZV9jb250cmFjdF9zaGEyNTYiOiAiNGM5YzNiY2ZmNWU5ZmQzMjUzMmQ3ZWRmODVkZjdhOTYxZTQ1ZWM5YzY0YmVlM2RmODgzM2Q5ODZhMmViMjc2ZSIsDQogICJ0aHJlc2hvbGRfYXBwbGljYXRpb25fd29ya2VyX3NoYTI1NiI6ICI1MDYxNmNiZDNmYzE0MDYzMjU3ODQyMWQ0NWMzYzQ1YTkwMjVjMDNjNGNmZmMyMThkNzlmMDIxMDM3NWFlM2VkIiwNCiAgInRlc3Rfd29ya2VyX3NoYTI1NiI6ICI3ZDIxZjk1YTY0NmQ1NjJiZmExMjRiOGYxODk4ZmVmOWRjODVmYmEyNmJlYzg5ZGMzZDg4NzUzZWU2ODI0MjcyIiwNCiAgInJ1bnRpbWVfbG9ja19tZW1iZXIiOiAiaW5mZXJlbmNlX3J1bnRpbWVfbG9jay5qc29uIiwNCiAgInJ1bnRpbWVfbG9ja19zaGEyNTYiOiAiNGM5YzNiY2ZmNWU5ZmQzMjUzMmQ3ZWRmODVkZjdhOTYxZTQ1ZWM5YzY0YmVlM2RmODgzM2Q5ODZhMmViMjc2ZSIsDQogICJkYXRhc2V0cyI6IHsNCiAgICAiZGV2aWduIjogew0KICAgICAgImV4cGVjdGVkIjogMjczMiwNCiAgICAgICJyb3dzIjogMjczMiwNCiAgICAgICJwcm9tcHRzX3NoYTI1NiI6ICI5MTdjYTE2NGRjYzIxNzgyM2FjYTdmNzE2MzQ4YTA2YTFmODJkOTU3YWVmYmFlMmZkY2EwMDQ3MmRlZWQwZWIxIiwNCiAgICAgICJwb29sX3NoYTI1NiI6ICIzMDZiM2MzMzkxYjIxNjQ5NjM1ZDE3NTdhZjM2NWZjOWRlNWZiOTI4ODk0OTg5YmZlMzgyZmQ2NDk4YTlmOGY5IiwNCiAgICAgICJ0aHJlc2hvbGRfbG9ja19zaGEyNTYiOiAiNTM3NjliNjUzZDUzYTMwZjUyZWQ4MTVjMzRjZTE1Nzc4NTYxZWNiNWUzM2Y5Y2Q0YjA5ZGM4ZTJjMWQ3ZDdmNCIsDQogICAgICAidGF1X3N0YXIiOiAwLjk5LA0KICAgICAgInRocmVzaG9sZF9zY29wZSI6ICJ2YWxpZGF0aW9uIiwNCiAgICAgICJ1bmlxdWVfaWRzIjogdHJ1ZSwNCiAgICAgICJub19sYWJlbF9maWVsZHMiOiB0cnVlLA0KICAgICAgImFsbF9wcm9tcHRfaGFzaGVzX3ZhbGlkIjogdHJ1ZSwNCiAgICAgICJiYXNlX3Byb21wdHNfc2hhMjU2IjogImE5MTIzMjJkOGRjNTU0YzJmMDQwMWFhMDlmMDAxNzZiZjIxOWE3OGUxMzUzOTgxYzlkNzkxMTg4NDU2ODQ2MTQiDQogICAgfSwNCiAgICAicmV2ZWFsIjogew0KICAgICAgImV4cGVjdGVkIjogMjI3NCwNCiAgICAgICJyb3dzIjogMjI3NCwNCiAgICAgICJwcm9tcHRzX3NoYTI1NiI6ICI3NDY1ODc5NWUwMzYyMzY4MDk0ZGM3NjZiNzIzMTQxMjUwYTAyNGZjOWJkZjA3NTNlZTE4ODcwYTExYTgyNDkyIiwNCiAgICAgICJwb29sX3NoYTI1NiI6ICIzN2JiMjRjNWJlMGUzNmM5ZjZjMDNhMDQ2ZTE3Njg3MjY1NDYxYWJlYjE4NzNmZTM5OWM5MjMyYzdlNzYxMjAzIiwNCiAgICAgICJ0aHJlc2hvbGRfbG9ja19zaGEyNTYiOiAiNjM1NTZlNWExMzUxMjViMDNiOTkzMWI4ZTllOTFlMDM1YWNmYjA5Yzg3ZmU1Njc0ZDI2MzFmNjlhNDNjODFkMiIsDQogICAgICAidGF1X3N0YXIiOiAwLjAxLA0KICAgICAgInRocmVzaG9sZF9zY29wZSI6ICJ2YWxpZGF0aW9uIiwNCiAgICAgICJ1bmlxdWVfaWRzIjogdHJ1ZSwNCiAgICAgICJub19sYWJlbF9maWVsZHMiOiB0cnVlLA0KICAgICAgImFsbF9wcm9tcHRfaGFzaGVzX3ZhbGlkIjogdHJ1ZSwNCiAgICAgICJiYXNlX3Byb21wdHNfc2hhMjU2IjogImRlNmM0MmNmZWNlYzhiZDdhMWZlNmIzMTNjODE2Yjc4NjY5NGEzOTk3OWU3YzY2ZWRjMGVlNzNiYjcwZmNjN2EiDQogICAgfQ0KICB9LA0KICAidGhyZXNob2xkX2xvY2tfc2hhMjU2Ijogew0KICAgICJkZXZpZ24iOiAiNTM3NjliNjUzZDUzYTMwZjUyZWQ4MTVjMzRjZTE1Nzc4NTYxZWNiNWUzM2Y5Y2Q0YjA5ZGM4ZTJjMWQ3ZDdmNCIsDQogICAgInJldmVhbCI6ICI2MzU1NmU1YTEzNTEyNWIwM2I5OTMxYjhlOWU5MWUwMzVhY2ZiMDljODdmZTU2NzRkMjYzMWY2OWE0M2M4MWQyIg0KICB9LA0KICAibGFiZWxzX2luX2thZ2dsZV9pbnB1dCI6IGZhbHNlLA0KICAicHJpdmF0ZV9sYWJlbF9zaGEyNTYiOiAiMzJkMjIzMjFhNGY2Y2QwMTRlNzZjZDc1YTJjZDU2OWE4MWM0ZGY0YjhhNjRiZjAzYjZjNGVkNTgyZTdlM2M3ZiIsDQogICJmcm96ZW5fc3BlY3NfemlwX3NoYTI1NiI6ICI5ZGIwZDNlZWJiNjZhMmY4M2MxZjk4MDNjZWIwMTkyOTU0YTNkZmUyYzNkZTk1MTUxODhhOTk2ZWRlODRlZDdiIiwNCiAgImthZ2dsZV9pbnB1dF9tZW1iZXJzX3NoYTI1NiI6IHsNCiAgICAiaW5mZXJlbmNlX3J1bnRpbWVfbG9jay5qc29uIjogIjRjOWMzYmNmZjVlOWZkMzI1MzJkN2VkZjg1ZGY3YTk2MWU0NWVjOWM2NGJlZTNkZjg4MzNkOTg2YTJlYjI3NmUiLA0KICAgICJwb29scy9kZXZpZ25fZWxpZ2libGVfcGFpcnMuanNvbmwiOiAiMzA2YjNjMzM5MWIyMTY0OTYzNWQxNzU3YWYzNjVmYzlkZTVmYjkyODg5NDk4OWJmZTM4MmZkNjQ5OGE5ZjhmOSIsDQogICAgInBvb2xzL3JldmVhbF9lbGlnaWJsZV9wYWlycy5qc29ubCI6ICIzN2JiMjRjNWJlMGUzNmM5ZjZjMDNhMDQ2ZTE3Njg3MjY1NDYxYWJlYjE4NzNmZTM5OWM5MjMyYzdlNzYxMjAzIiwNCiAgICAidGhyZXNob2xkX2xvY2tzL2Rldmlnbl90aHJlc2hvbGRfbG9jay5qc29uIjogIjUzNzY5YjY1M2Q1M2EzMGY1MmVkODE1YzM0Y2UxNTc3ODU2MWVjYjVlMzNmOWNkNGIwOWRjOGUyYzFkN2Q3ZjQiLA0KICAgICJ0aHJlc2hvbGRfbG9ja3MvcmV2ZWFsX3RocmVzaG9sZF9sb2NrLmpzb24iOiAiNjM1NTZlNWExMzUxMjViMDNiOTkzMWI4ZTllOTFlMDM1YWNmYjA5Yzg3ZmU1Njc0ZDI2MzFmNjlhNDNjODFkMiIsDQogICAgInByb21wdHMvZGV2aWduLmpzb25sIjogIjkxN2NhMTY0ZGNjMjE3ODIzYWNhN2Y3MTYzNDhhMDZhMWY4MmQ5NTdhZWZiYWUyZmRjYTAwNDcyZGVlZDBlYjEiLA0KICAgICJwcm9tcHRzL3JldmVhbC5qc29ubCI6ICI3NDY1ODc5NWUwMzYyMzY4MDk0ZGM3NjZiNzIzMTQxMjUwYTAyNGZjOWJkZjA3NTNlZTE4ODcwYTExYTgyNDkyIg0KICB9LA0KICAia2FnZ2xlX2lucHV0X2FyY2hpdmVfc2hhMjU2IjogIjhiOTQ1OGRmYWJmNTY5MDZiYmJmMjM0MWMwZjdjOGZjOWU4NTY3NmU3YzFmMDYzMzA1YTk1NjIwNWRjYTRjYzAiLA0KICAibm90ZXMiOiAiVGVzdCBsYWJlbHMgcmVtYWluIGxvY2FsOyB0aHJlc2hvbGQgbG9ja3Mgd2VyZSBmaXR0ZWQgb24gdmFsaWRhdGlvbiBvbmx5LiBObyB0aHJlc2hvbGQgcmUtZml0IG9uIHRlc3QuIiwNCiAgInRocmVzaG9sZF9mcm9tIjogIkIzIiwNCiAgImJhc2VfcmVsZWFzZV9tYW5pZmVzdF9zaGEyNTYiOiAiYTI1YTVjZWE4MTc2MGI0MGJmMmNkZTcwYjE0NWQ1MGY2ODg4MGI5NzYzOWNhYzczZjhkY2I3ZmVhMDkxYmJmMSIsDQogICJzaGFyZWRfd29ya2VyX3NoYTI1NiI6IHsNCiAgICAiZ3JhY2VfcnVudGltZV9jb250cmFjdC5weSI6ICI0MzlkMzFiZmU0MzI0Yjc5ZDliMTE5ZDcxNzhmYTI5MGJhYzY1MWY0YmUzMzZhMTIxNzU3M2JmZDg2YmQ1NDc3IiwNCiAgICAiYjNfbmF0aXZlX3J1bnRpbWUucHkiOiAiOTYwZDNlMzU1ODllYmQ2ZGMzMWYwYTcwZGI5ZjgwOTYxNGFmZGI5MmUwMTMyYzkxZTI3NzE4MzgzZjIxMzg2ZiIsDQogICAgImthZ2dsZV9pbmZlcmVuY2VfdGVzdC5weSI6ICI3ZDIxZjk1YTY0NmQ1NjJiZmExMjRiOGYxODk4ZmVmOWRjODVmYmEyNmJlYzg5ZGMzZDg4NzUzZWU2ODI0MjcyIiwNCiAgICAiYjNfdGVzdF9ydW50aW1lLnB5IjogImRmZDE3ZmYwMWNmN2I0ZDhhNTQ5Zjk2MzcxYWM0NTA3OGQ0YTE2NDQwMGMwNWUyNTc0YzlmYjQxN2FiMTcyNDAiLA0KICAgICJpbmZlcmVuY2VfcnVudGltZV9sb2NrLmpzb24iOiAiNGM5YzNiY2ZmNWU5ZmQzMjUzMmQ3ZWRmODVkZjdhOTYxZTQ1ZWM5YzY0YmVlM2RmODgzM2Q5ODZhMmViMjc2ZSINCiAgfQ0KfQ0K'
INPUT_ROOT=Path('/kaggle/input')
DATASETS_ROOT=INPUT_ROOT/'datasets'
input_roots=sorted(p for p in INPUT_ROOT.iterdir() if p.is_dir())

def walk_tree(root):
 visited=set()
 if not root.exists(): return
 for current,dirs,files in os.walk(root,topdown=True,followlinks=True):
  real=os.path.realpath(current)
  if real in visited:
   dirs[:]=[]
   continue
  visited.add(real)
  dirs.sort()
  yield Path(current),dirs,files

manifest_candidates=set()
for root in input_roots:
 for current,dirs,files in walk_tree(root):
  if 'test_release_manifest.json' in files:
   manifest_candidates.add((current/'test_release_manifest.json').resolve())
manifest_candidates=sorted(manifest_candidates)
if len(manifest_candidates)>1:
 raise RuntimeError('Found multiple B3 test manifests: '+json.dumps([str(p) for p in manifest_candidates]))
if manifest_candidates:
 MANIFEST_PATH=manifest_candidates[0]
 MANIFEST_SOURCE='mounted_sidecar'
else:
 MANIFEST_PATH=WORK/'embedded_test_release_manifest.json'
 embedded=base64.b64decode(EMBEDDED_MANIFEST_B64)
 if MANIFEST_PATH.exists() and MANIFEST_PATH.read_bytes()!=embedded:
  raise RuntimeError('Existing embedded test manifest differs from sealed release')
 MANIFEST_PATH.write_bytes(embedded)
 MANIFEST_SOURCE='sealed_manifest_embedded_in_notebook'
if hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()!=EXPECTED_MANIFEST_SHA256:
 raise RuntimeError('Test release manifest checksum mismatch')
from b3_test_runtime import load_manifest, read_input_directory, validate_members, validate_archive, read_dataset_rows, load_threshold
ROOT=load_manifest(MANIFEST_PATH,EXPECTED_MANIFEST_SHA256)
EXPECTED_MEMBERS=set(ROOT['kaggle_input_members_sha256'])
def has_expected_members(folder):
 folder=Path(folder)
 return folder.is_dir() and all((folder/name).is_file() for name in EXPECTED_MEMBERS)

expanded_candidates=set()
archive_candidates=set()
for current,dirs,files in walk_tree(DATASETS_ROOT):
 current=Path(current)
 if 'inference_runtime_lock.json' in files and has_expected_members(current):
  expanded_candidates.add(current.resolve())
 for filename in files:
  if filename.lower().endswith('.zip'):
   archive_candidates.add((current/filename).resolve())
expanded_candidates=sorted(expanded_candidates)
archive_candidates=sorted(p for p in archive_candidates if digest(p)==ROOT['kaggle_input_archive_sha256'])
if len(expanded_candidates)==1:
 INPUT_DIR=expanded_candidates[0]
 MEMBERS=validate_members(read_input_directory(INPUT_DIR),ROOT)
 INPUT_SOURCE=INPUT_DIR
elif len(expanded_candidates)>1:
 raise RuntimeError('Found multiple expanded B3 input bundles: '+json.dumps([str(p) for p in expanded_candidates]))
elif len(archive_candidates)==1:
 archive_input=archive_candidates[0]
 MEMBERS=validate_archive(archive_input,ROOT)
 INPUT_DIR=WORK/'b3_test_input_expanded'
 if INPUT_DIR.exists():
  existing=read_input_directory(INPUT_DIR)
  validate_members(existing,ROOT)
 else:
  INPUT_DIR.mkdir(parents=True)
  for member_name,member_bytes in MEMBERS.items():
   parts=member_name.split('/')
   if not parts or any(part in ('','.','..') for part in parts) or chr(92) in member_name or ':' in member_name:
    raise RuntimeError('Unsafe path in validated test archive: '+member_name)
   target=INPUT_DIR.joinpath(*parts)
   target.parent.mkdir(parents=True,exist_ok=True)
   target.write_bytes(member_bytes)
 MEMBERS=validate_members(read_input_directory(INPUT_DIR),ROOT)
 INPUT_SOURCE=archive_input
else:
 details={'datasets_root':str(DATASETS_ROOT),'manifest_source':MANIFEST_SOURCE,'manifest_path':str(MANIFEST_PATH),'expanded_candidates':[str(p) for p in expanded_candidates],'archive_candidates':[str(p) for p in archive_candidates],'mounted_roots':{str(p):sorted(x.name for x in p.iterdir())[:25] for p in input_roots}}
 print(json.dumps({'preflight':'FAIL',**details},indent=2))
 raise RuntimeError('Could not locate one expanded B3 bundle or kaggle_input.zip')
model_filename=RUNTIME['model']['filename']
model_candidates=set()
for root in input_roots:
 for current,dirs,files in walk_tree(root):
  if model_filename in files:
   model_candidates.add((current/model_filename).resolve())
model_candidates=sorted(model_candidates)
if len(model_candidates)==1:
 MODEL_PATH=model_candidates[0]
else:
 print(json.dumps({'preflight':'FAIL','manifest':str(MANIFEST_PATH),'input_source':str(INPUT_SOURCE),'model_filename':model_filename,'model_candidates':[str(x) for x in model_candidates],'input_roots':[str(x) for x in input_roots]},indent=2))
 raise RuntimeError('Could not resolve exactly one pinned model; see candidate paths above')
print(json.dumps({'preflight':'PASS','manifest_source':MANIFEST_SOURCE,'manifest':str(MANIFEST_PATH),'input_source':str(INPUT_SOURCE),'input_mode':'kaggle_auto_extracted' if INPUT_DIR.is_dir() and INPUT_DIR in expanded_candidates else 'verified_archive','model_path':str(MODEL_PATH),'model_candidate_count':len(model_candidates)},indent=2))
if hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()!=RUNTIME['model']['sha256']: raise RuntimeError('Model checkpoint checksum mismatch')
if hashlib.sha256(Path('/kaggle/working/kaggle_inference_test.py').read_bytes()).hexdigest()!=ROOT['test_worker_sha256']: raise RuntimeError('Test worker source hash mismatch')
if hashlib.sha256(MEMBERS[ROOT['runtime_lock_member']]).hexdigest()!=ROOT['runtime_lock_sha256']: raise RuntimeError('Runtime contract hash mismatch')
if ROOT['runtime_lock_sha256']!=hashlib.sha256(Path('/kaggle/working/inference_runtime_lock.json').read_bytes()).hexdigest(): raise RuntimeError('Notebook/runtime lock differs from release lock')
for ds in ('devign','reveal'):
 rows=read_dataset_rows(MEMBERS,ROOT,ds)
 tau=load_threshold(ROOT,MEMBERS,ds)
 print(f'Preflight {ds}: n={len(rows)} threshold={tau}')
gpu=command(['nvidia-smi','--query-gpu=index,uuid,name,memory.free,memory.total,driver_version','--format=csv,noheader'])
if len([x for x in gpu.splitlines() if x.strip()])!=RUNTIME['gpu']['device_count'] or any('T4' not in x for x in gpu.splitlines()): raise RuntimeError('Runtime contract requires 2x Tesla T4')

for name, expected in ROOT['shared_worker_sha256'].items():
 if digest(WORK/name)!=expected: raise RuntimeError('Shared worker hash mismatch: '+name)
if ROOT['arm'] != 'B3-swap': raise RuntimeError('Notebook arm differs from manifest')


{
  "preflight": "PASS",
  "manifest_source": "sealed_manifest_embedded_in_notebook",
  "manifest": "/kaggle/working/embedded_test_release_manifest.json",
  "input_source": "/kaggle/input/datasets/duchuynguyen1802/b3-swap-kaggle-input-zip",
  "input_mode": "kaggle_auto_extracted",
  "model_path": "/kaggle/input/notebooks/duchuynguyen1802/huy-qwen3coder30b-prepare-notebook/qwen_gguf/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf",
  "model_candidate_count": 1
}
Preflight devign: n=2732 threshold=0.99
Preflight reveal: n=2274 threshold=0.01


In [3]:
import importlib.metadata, sys, urllib.request, shutil, os
build=RUNTIME['build']
os.environ['CMAKE_ARGS']=(f"-DGGML_CUDA={'on' if build['GGML_CUDA'] else 'off'}" f" -DCMAKE_CUDA_ARCHITECTURES={build['CMAKE_CUDA_ARCHITECTURES']}" f" -DCMAKE_LIBRARY_PATH={build['CMAKE_LIBRARY_PATH']}")
for key in ('CMAKE_BUILD_PARALLEL_LEVEL','FORCE_CMAKE'): os.environ[key]=build[key]
packages=[f'{n}=={v}' for n,v in RUNTIME['packages'].items() if n!=RUNTIME['backend_source']['name']]
packages += ['typing-extensions>=4.5.0','diskcache>=5.6.1','jinja2>=2.11.3']
diag=Path('/kaggle/working/grace_diagnostics'); diag.mkdir(exist_ok=True)
with (diag/'install_dependencies.log').open('w',encoding='utf-8') as log:
 subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir',*packages],check=True,stdout=log,stderr=subprocess.STDOUT)
source=RUNTIME['backend_source']
with urllib.request.urlopen(f"https://pypi.org/pypi/{source['name']}/{source['version']}/json",timeout=60) as response: meta=json.load(response)
artifact=next(x for x in meta['urls'] if x['filename']==source['sdist_filename'])
if artifact['digests']['sha256']!=source['sdist_sha256']: raise RuntimeError('llama.cpp source registry hash mismatch')
archive=Path('/kaggle/working')/source['sdist_filename']
with urllib.request.urlopen(artifact['url'],timeout=120) as response, archive.open('wb') as target: shutil.copyfileobj(response,target)
if digest(archive)!=source['sdist_sha256']: raise RuntimeError('llama.cpp source archive hash mismatch')
with (diag/'install_llama_cuda.log').open('w',encoding='utf-8') as log:
 subprocess.run([sys.executable,'-m','pip','install','-v','--no-cache-dir','--no-deps','--force-reinstall',str(archive)],check=True,stdout=log,stderr=subprocess.STDOUT)
assert_runtime(RUNTIME)


({'huggingface-hub': '0.36.2',
  'llama-cpp-python': '0.3.16',
  'numpy': '2.1.3',
  'scikit-learn': '1.5.2',
  'scipy': '1.16.3',
  'tokenizers': '0.22.2',
  'transformers': '4.57.1'},
 '0, GPU-1748faac-e30c-8cf6-2624-d771e2c02d11, Tesla T4, 14912 MiB, 15360 MiB, 580.178.04\n1, GPU-dba30d94-b23c-2cb2-6c37-32e6250fa4ba, Tesla T4, 14912 MiB, 15360 MiB, 580.178.04\n')

In [4]:
assert_runtime(RUNTIME)
import llama_cpp
assert llama_cpp.llama_supports_gpu_offload(), 'Locked CUDA backend is unavailable'
print('Locked package versions:', {n:importlib.metadata.version(n) for n in RUNTIME['packages']})
print(command(['nvidia-smi','--query-gpu=index,uuid,name,memory.free,memory.total,driver_version','--format=csv,noheader']))


ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 2 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
  Device 1: Tesla T4, compute capability 7.5, VMM: yes


Locked package versions: {'huggingface-hub': '0.36.2', 'llama-cpp-python': '0.3.16', 'numpy': '2.1.3', 'scikit-learn': '1.5.2', 'scipy': '1.16.3', 'tokenizers': '0.22.2', 'transformers': '4.57.1'}
0, GPU-1748faac-e30c-8cf6-2624-d771e2c02d11, Tesla T4, 14909 MiB, 15360 MiB, 580.178.04
1, GPU-dba30d94-b23c-2cb2-6c37-32e6250fa4ba, Tesla T4, 14807 MiB, 15360 MiB, 580.178.04



In [5]:
import os, sys, time, subprocess, shutil, json
from pathlib import Path
from b3_test_runtime import checked_records
OUT=Path('/kaggle/working/b3_swap_test_release_20261110')
INPUT_ROOT=Path('/kaggle/input')
def find_checkpoint_roots():
    found=[]
    for root in sorted(p for p in INPUT_ROOT.iterdir() if p.is_dir()):
        for current,dirs,files in walk_tree(root):
            current=Path(current)
            if 'status.json' not in files: continue
            status_file=current/'status.json'
            try:
                status_data=json.loads(status_file.read_text(encoding='utf-8'))
                if status_data.get('state')!='timed_out' or status_data.get('timed_out') is not True: continue
                if status_data.get('runs',{}).get('reveal/test',{}).get('state')!='checkpointed': continue
                if status_data.get('runs',{}).get('devign/test',{}).get('state')!='complete': continue
                dn=checked_records(current,ROOT,read_dataset_rows(MEMBERS,ROOT,'devign'),'devign')
                rn=checked_records(current,ROOT,read_dataset_rows(MEMBERS,ROOT,'reveal'),'reveal')
                if dn!=ROOT['datasets']['devign']['expected'] or not 0<rn<ROOT['datasets']['reveal']['expected']: continue
                found.append((current.resolve(),status_data,dn,rn))
            except Exception as exc:
                print(f'Skipping non-matching checkpoint candidate {status_file}: {type(exc).__name__}: {exc}')
    return found
if OUT.exists():
    if not (OUT/'status.json').is_file(): raise RuntimeError(f'Output path exists without resumable status: {OUT}')
    prior=json.loads((OUT/'status.json').read_text(encoding='utf-8'))
    if prior.get('state') not in ('timed_out','checkpointed','running'): raise RuntimeError(f'Existing output is not resumable: {prior.get("state")}')
    dn=checked_records(OUT,ROOT,read_dataset_rows(MEMBERS,ROOT,'devign'),'devign')
    rn=checked_records(OUT,ROOT,read_dataset_rows(MEMBERS,ROOT,'reveal'),'reveal')
    if dn!=ROOT['datasets']['devign']['expected']: raise RuntimeError(f'Existing Devign checkpoint incomplete: {dn}')
    print(f'Resuming verified working checkpoint: Devign {dn}; REVEAL {rn}/{ROOT["datasets"]["reveal"]["expected"]}')
else:
    candidates=find_checkpoint_roots()
    summary=[{'path':str(p),'devign':d,'reveal':r} for p,_,d,r in candidates]
    print(json.dumps({'checkpoint_discovery':'verified','candidate_count':len(candidates),'candidates':summary},indent=2))
    if len(candidates)!=1: raise FileNotFoundError(f'Expected exactly one verified prior B3-swap checkpoint under {INPUT_ROOT}; found {len(candidates)}. Attach the previous notebook version as an input.')
    checkpoint,old_status,dn,rn=candidates[0]
    shutil.copytree(checkpoint,OUT)
    print(f'Copied verified checkpoint: {checkpoint}')
    print(f'Checkpoint census: Devign {dn}/{ROOT["datasets"]["devign"]["expected"]}; REVEAL {rn}/{ROOT["datasets"]["reveal"]["expected"]}')
env=os.environ.copy()
env.update(B3_TEST_MANIFEST_PATH=str(MANIFEST_PATH),B3_TEST_MANIFEST_SHA256=EXPECTED_MANIFEST_SHA256,B3_TEST_INPUT_DIR=str(INPUT_DIR))
status=OUT/'status.json'
cmd=[sys.executable,'/kaggle/working/kaggle_inference_test.py','--model',str(MODEL_PATH),'--seed','42','--arm','B3-swap','--expected-model-sha256',RUNTIME['model']['sha256'],'--model-revision','local-input','--gpu-layers','-1','--deadline-epoch',str(SESSION_STARTED+32400),'--checkpoint-reserve-seconds','300','--status',str(status),'--run','devign','test','devign',str(OUT/'seed_42/devign/test'),'--run','reveal','test','reveal',str(OUT/'seed_42/reveal/test')]
print(f'Launching resume runner; deadline epoch={SESSION_STARTED+32400:.0f}; it skips IDs already present in verified records.jsonl')
subprocess.run(cmd,check=True,env=env)
final_status=json.loads(status.read_text(encoding='utf-8'))
print(json.dumps({'final_state':final_status.get('state'),'timed_out':final_status.get('timed_out'),'completed_records_total':final_status.get('completed_records_total'),'runs':final_status.get('runs')},indent=2))


{
  "checkpoint_discovery": "verified",
  "candidate_count": 1,
  "candidates": [
    {
      "path": "/kaggle/input/notebooks/duchuynguyen1802/b3-swap-test-latest-ipynb/b3_swap_test_release_20261110",
      "devign": 2732,
      "reveal": 1865
    }
  ]
}
Copied verified checkpoint: /kaggle/input/notebooks/duchuynguyen1802/b3-swap-test-latest-ipynb/b3_swap_test_release_20261110
Checkpoint census: Devign 2732/2732; REVEAL 1865/2274
Launching resume runner; deadline epoch=1794399096; it skips IDs already present in verified records.jsonl


ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 2 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
  Device 1: Tesla T4, compute capability 7.5, VMM: yes


llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) - 14806 MiB free
llama_model_load_from_file_impl: using device CUDA1 (Tesla T4) - 14704 MiB free
llama_model_loader: loaded meta data with 39 key-value pairs and 579 tensors from /kaggle/input/notebooks/duchuynguyen1802/huy-qwen3coder30b-prepare-notebook/qwen_gguf/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3moe
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_mode